In [1]:
import gymnasium as gym
import numpy as np

In [ ]:
# Initialize the environment and get its information
env = gym.make('FrozenLake-v1', is_slippery=True)
obs_space = env.observation_space
n_states = obs_space.n
action_space = env.action_space
n_actions = action_space.n
P = env.unwrapped.P # Transition Dynamics


In [8]:
print("Observation Space:", obs_space)
print("Total Number of States:", n_states)

# Check the actions
print("\nAction Space:", action_space)
print("Total Number of Actions:", n_actions)

Observation Space: Discrete(16)
Total Number of States: 16

Action Space: Discrete(4)
Total Number of Actions: 4


* **The Grid:** The agent starts at the top left of a 4x4 grid (a frozen lake) and must reach a frisbee at the bottom right.

* **The Hazards:** The lake has several thin ice holes scattered across it. Falling into a hole instantly ends the episode.

* **The Slippery Rule:** When you tell the agent to move a certain direction, it only has a 33% chance of actually going that way, and a 33% chance each of slipping into the two perpendicular directions (you can never slip backwards).

* **The Rewards:** Reaching the goal is **+1**. Every other step—and falling into a hole—is **0**. (This is called a "sparse reward" because the agent gets zero feedback until the very end).

* **The Space:** 16 states (for the 4x4 version), 4 actions (Up, Down, Left, Right). 

This setup is a defined Markov Decision Process (MDP) because it satisfies the Markov Property: the future state of the agent depends only on the current state and the chosen action, not on the sequence of events that preceded it. 

Furthermore, the environment is fully observable (the agent knows exactly what square it is on), and the transition probabilities ($P$) and rewards ($R$) are fixed and mathematical.

In [ ]:
def value_iteration(P, n_states, n_actions, gamma):
    """
    Computes the optimal Value function for the MDP.
    """
    theta=1e-8 # The threshold for stopping the loop (when values stop changing)
    # Initialize the Value table
    V = np.zeros(n_states)
    
    # We want to calculate the optimal move for every single square on the entire board
    # to try to find the best path
    while True:
        delta = 0
        # Loop through every single state in the grid
        for s in range(n_states):
            # We want to find the highest value among all possible actions
            v_best = -float('inf')
            
            for a in range(n_actions):
                action_value = 0
                
                # The environment dictionary P[s][a] returns a list of possible outcomes:
                # (probability, next_state, reward, is_terminated)
                for prob, next_state, reward, done in P[s][a]:
                    # The core Bellman Equation calculation
                    # If the game is done, future value is 0
                    future_val = V[next_state] if not done else 0.0
                    action_value += prob * (reward + gamma * future_val)
                    
                # Keep the highest action value
                if action_value > v_best:
                    v_best = action_value
                    
            # Check how much the value changed
            delta = max(delta, abs(V[s] - v_best))
            # Update the Value table
            V[s] = v_best
            
        # If the table barely changed, we have solved the MDP
        if delta < theta:
            break
            
    return V

In [16]:
def extract_policy(V, P, n_states, n_actions, gamma):
    """
    Once we have the optimal Value table, we extract the best policy (moves).
    """
    policy = np.zeros(n_states, dtype=int)
    
    for s in range(n_states):
        best_action = 0
        best_value = -float('inf')
        
        for a in range(n_actions):
            action_value = 0
            for prob, next_state, reward, done in P[s][a]:
                future_val = V[next_state] if not done else 0.0
                action_value += prob * (reward + gamma * future_val)
                
            if action_value > best_value:
                best_value = action_value
                best_action = a
                
        policy[s] = best_action
        
    return policy

In [12]:
def print_value_function(V):
    print("Value Function:")
    print(np.round(V, 3).reshape(4, 4))

def print_policy(policy, ):
    # Map actions to arrows: 0:Left, 1:Down, 2:Right, 3:Up
    action_mapping = {0: '←', 1: '↓', 2: '→', 3: '↑'}
    grid = [action_mapping[action] for action in policy]
    
    # Manually overwrite the Holes (H) and Goal (G) for visual clarity
    holes = [5, 7, 11, 12]
    for h in holes: 
        grid[h] = 'H'
    grid[15] = 'G'
    
    print("\nOptimal Policy:")
    grid = np.array(grid).reshape(4, 4)
    for row in grid:
        print(" ".join(row))

In [17]:
# Discount factor gamma
# If gamma = 1.0: Perfectly Far-sighted, cares about future rewards
# gamma = 0 : only cares about immediate next step ==> not good
optimal_V = value_iteration(P, n_states, n_actions, gamma=0.99)
optimal_policy = extract_policy(optimal_V, P, n_states, n_actions, gamma=0.99)

In [18]:
print_value_function(optimal_V)
print_policy(optimal_policy)

Value Function:
[[0.542 0.499 0.471 0.457]
 [0.558 0.    0.358 0.   ]
 [0.592 0.643 0.615 0.   ]
 [0.    0.742 0.863 0.   ]]

Optimal Policy:
← ↑ ↑ ↑
← H ← H
↑ ↓ ← H
H → ↓ G


We can see the agent learned the Wall-bouncing strategy where it walks face-first into a boundary wall to block its own forward movement. By pointing itself away from danger, it ensures that even if it slips, it will only hit a wall or a safe square.

**Example Row 0, Column 1 where the policy tells it to move upward:**

33% chance (Intended): It moves upward, bumps the wall, and stays safe there.

33% chance (Slip 1): It slips to the right, successfully sliding to the square on its right.

33% chance (Slip 2): It slips to the left, successfully sliding to the starting square.

There is a 0% chance of moving down, where the hole is.

The numbers in the value function represent the "Expected Future Reward" of a square, meaning higher is better. The agent's policy is simply to look around and move toward the highest number.

Terminal states (the Goal and the Holes) have a mathematical value of 0.0 because the game instantly ends there, offering no future moves. Because the goal is zero, the highest value on the board (0.863) belongs to the square directly next to the goal. The values smoothly ripple backward from this launchpad all the way to the start (0.542), which creates a path for the agent to follow.

In [19]:
short_sighted_V = value_iteration(P, n_states, n_actions, gamma=0.1)
short_sighted_policy = extract_policy(short_sighted_V, P, n_states, n_actions, gamma=0.1)

In [20]:
print_value_function(short_sighted_V)
print_policy(short_sighted_policy)

Value Function:
[[0.    0.    0.    0.   ]
 [0.    0.    0.    0.   ]
 [0.    0.001 0.012 0.   ]
 [0.    0.012 0.345 0.   ]]

Optimal Policy:
↓ ↑ → ↑
← H ← H
↑ ↓ ← H
H → ↓ G


Since the discount factor (γ) is so low, the value of the distant +1 reward decays to zero almost immediately. Consequently, the top half of the grid is completely filled with zeros because the agent cannot mathematically "see" the reward from that far away. The agent fails to find a good solution from the starting square.